In [1]:
import os
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
import gradio as gr
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

c:\Users\DELL\Documents\Programming\Python\Agentic\TenderWiseAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from agents import (
    Agent, ModelSettings, RunContextWrapper, Runner, function_tool,
    set_default_openai_api, set_tracing_disabled,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

In [3]:
try:
    import pymupdf
except Exception:
    pymupdf = None

In [4]:
load_dotenv(override=True)
token = os.getenv("API_TOKEN")
if not token:
    raise ValueError(
        "API_TOKEN not found. Check your .env file name and location.")

In [5]:
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

In [ ]:
MODEL_NAME = "nvidia/nemotron-3-ultra-550b-a55b:free"

openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=token)
MODEL = OpenAIChatCompletionsModel(
    model=MODEL_NAME, openai_client=openrouter_client)

MAX_TENDER_CHARS = 200_000

In [7]:
class Requirement(BaseModel):
    description: str = Field(description="The requirement in plain words")
    mandatory: bool = Field(
        description="True only if the tender uses words like must, shall, required")
    source_page: str = Field(
        description='Page number from the === PAGE n === markers, or "Not stated"')
    source_quote: str = Field(
        description="Short exact quote (max 25 words) copied word for word from the tender")


class Deadline(BaseModel):
    label: str
    date_text: str = Field(
        description="Date and time exactly as written in the tender")
    source_page: str
    source_quote: str


class TenderAnalysis(BaseModel):
    tender_title: str
    issuing_organisation: str
    reference_number: str
    summary: str = Field(
        description="3 to 5 neutral sentences on what is being bought and by whom")
    deadlines: list[Deadline]
    eligibility: list[Requirement]
    mandatory_documents: list[Requirement]
    technical_requirements: list[Requirement]
    commercial_requirements: list[Requirement]
    evaluation_criteria: list[str] = Field(
        description="How bids are scored, including weights if stated")
    submission_instructions: list[str]
    unclear_or_missing_info: list[str] = Field(
        description="Important things the tender does NOT say or says vaguely")

In [8]:
ComplianceStatus = Literal["evidence_supports",
                           "partially_supported", "missing_evidence", "not_met", "unclear"]


class ComplianceItem(BaseModel):
    requirement: str
    status: ComplianceStatus
    evidence: str = Field(
        description="Short exact quote from the company details, or empty text if none")
    action_needed: str = Field(
        description="What the company must do or attach, or 'None'")


class GapAssessment(BaseModel):
    items: list[ComplianceItem]
    overall_readiness: Literal["low", "medium", "high", "cannot_assess"]
    summary: str
    clarification_questions: list[str] = Field(
        description="Questions to send to the tender issuer")

In [9]:
# ---------- Risk Analyst ----------
class Risk(BaseModel):
    title: str
    description: str
    severity: Literal["low", "medium", "high"]
    origin: Literal["stated_in_tender", "ai_inference"]
    source_page: str
    recommendation: str


class RiskAssessment(BaseModel):
    risks: list[Risk]


# ---------- Solution Advisor ----------
class Recommendation(BaseModel):
    title: str
    detail: str
    addresses: str = Field(
        description="Which tender requirement, risk or scoring point this helps with")
    needs_from_company: str = Field(
        description="What the company must confirm or provide, or 'None'")


class SolutionAdvice(BaseModel):
    technical: list[Recommendation]
    commercial: list[Recommendation]
    teaming_or_subcontracting: list[Recommendation]
    assumptions: list[str]

In [10]:
# The safety checks
def _norm(text: str) -> str:
    """Lowercase and drop punctuation and extra spaces, so line breaks don't matter."""
    return re.sub(r"[\W_]+", " ", text.lower()).strip()


def quote_in_text(quote: str, text: str) -> bool:
    """True if every piece of the quote (split on '...') appears in the text."""
    haystack = _norm(text)
    parts = [p for p in re.split(r"\.\.\.|\u2026", quote) if _norm(p)]
    if not parts:
        return False
    return all(len(_norm(p)) >= 8 and _norm(p) in haystack for p in parts)


def find_unverified_quotes(analysis: TenderAnalysis, tender_text: str) -> list[str]:
    groups = [analysis.eligibility, analysis.mandatory_documents,
              analysis.technical_requirements, analysis.commercial_requirements]
    missing = []
    for group in groups:
        for req in group:
            if not quote_in_text(req.source_quote, tender_text):
                missing.append(f"{req.description} (page {req.source_page})")
    for d in analysis.deadlines:
        if not quote_in_text(d.source_quote, tender_text):
            missing.append(f"Deadline: {d.label} (page {d.source_page})")
    return missing


def enforce_evidence_rules(assessment: GapAssessment, profile_text: str) -> GapAssessment:
    """Downgrade any 'supported' claim that is not backed by a real quote from the company details."""
    fixed = assessment.model_copy(deep=True)

    if not profile_text.strip():
        for item in fixed.items:
            item.status = "missing_evidence"
            item.evidence = ""
        fixed.overall_readiness = "cannot_assess"
        fixed.summary = "No company details were supplied, so compliance cannot be assessed. " + fixed.summary
        return fixed

    for item in fixed.items:
        if item.status in ("evidence_supports", "partially_supported"):
            if not item.evidence.strip():
                item.status = "missing_evidence"
            elif not quote_in_text(item.evidence, profile_text):
                item.status = "unclear"
                item.action_needed = "Quoted evidence could not be matched to the company details. Check by hand. " + item.action_needed

    if fixed.overall_readiness == "high" and any(i.status != "evidence_supports" for i in fixed.items):
        fixed.overall_readiness = "medium"
    return fixed

In [11]:
# The reading helpers
def read_pages(path: Path) -> list[str]:
    suffix = path.suffix.lower()
    if suffix == ".txt":
        return [path.read_text(encoding="utf-8", errors="replace")]
    if suffix == ".pdf":
        if pymupdf is not None:
            with pymupdf.open(str(path)) as doc:
                return [page.get_text() for page in doc]
        from pypdf import PdfReader
        return [(p.extract_text() or "") for p in PdfReader(str(path)).pages]
    if suffix == ".docx":
        from docx import Document
        doc = Document(str(path))
        parts = [p.text for p in doc.paragraphs]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text.strip()
                             for cell in row.cells))
        return ["\n".join(parts)]
    raise ValueError(
        f"Unsupported file type '{suffix}'. Please use a PDF, Word (.docx) or TXT file.")


def pages_to_text(pages: list[str], max_chars: int):
    """Join pages with markers. Returns (text, was_cut, pages_included)."""
    parts, total, included = [], 0, 0
    for number, text in enumerate(pages, start=1):
        block = f"=== PAGE {number} ===\n{text.strip()}\n"
        if total + len(block) > max_chars and parts:
            return "\n".join(parts), True, included
        parts.append(block)
        total += len(block)
        included += 1
    return "\n".join(parts), False, included

In [12]:
# The shared notebook and the Intake Agent's tools
@dataclass
class TenderState:
    tender_text: str = ""
    tender_source: str = ""
    pages: int = 0
    truncated: bool = False
    company_profile: str = ""
    # file_id -> path of the uploaded file
    uploads: dict = field(default_factory=dict)
    # filled in as specialists finish
    results: dict = field(default_factory=dict)


@function_tool
def load_tender_file(ctx: RunContextWrapper[TenderState], file_id: str) -> str:
    """Load the tender document that the user attached in the chat (PDF, Word or TXT).

    Args:
        file_id: The file_id from the note about the attachment, for example upload_1
    """
    path = ctx.context.uploads.get(file_id.strip())
    if path is None:
        return f"There is no attachment called '{file_id}'. Ask the user to attach the file again."
    try:
        text, truncated, included = pages_to_text(
            read_pages(path), MAX_TENDER_CHARS)
    except Exception as e:
        return f"Could not read the file: {e}"
    if len(re.sub(r"=== PAGE \d+ ===", "", text).strip()) < 200:
        return "Almost no text was found. This looks like a scanned PDF. Ask for a text-based PDF, a Word file or a TXT version."
    ctx.context.tender_text = text
    ctx.context.tender_source = path.name
    ctx.context.pages = included
    ctx.context.truncated = truncated
    note = " The tender was long, so it was cut at the size limit." if truncated else ""
    return f"Loaded {path.name}: {included} pages, {len(text):,} characters.{note}"


@function_tool
def save_tender_text(ctx: RunContextWrapper[TenderState], text: str) -> str:
    """Save tender details the user typed in the chat, such as a summary of the opportunity.

    Args:
        text: The tender details exactly as the user gave them
    """
    ctx.context.tender_text += f"\n=== USER-PROVIDED TENDER NOTES ===\n{text.strip()}\n"
    if not ctx.context.tender_source:
        ctx.context.tender_source = "typed notes"
    return "Saved the tender notes."


@function_tool
def save_company_details(ctx: RunContextWrapper[TenderState], details: str) -> str:
    """Save facts about the company that the user told you. Use the user's own words. Never add, improve or guess facts.

    Args:
        details: The facts exactly as the user stated them
    """
    ctx.context.company_profile += f"- {details.strip()}\n"
    return "Saved."


@function_tool
def load_company_file(ctx: RunContextWrapper[TenderState], file_id: str) -> str:
    """Load a company profile or capability statement that the user attached in the chat (PDF, Word or TXT).

    Args:
        file_id: The file_id from the note about the attachment, for example upload_2
    """
    path = ctx.context.uploads.get(file_id.strip())
    if path is None:
        return f"There is no attachment called '{file_id}'. Ask the user to attach the file again."
    try:
        text = "\n".join(read_pages(path)).strip()[:30_000]
    except Exception as e:
        return f"Could not read the file: {e}"
    if not text:
        return "No text could be read from that file."
    ctx.context.company_profile += f"\n[From file {path.name}]\n{text}\n"
    return f"Loaded company details from {path.name} ({len(text):,} characters)."


@function_tool
def check_intake_status(ctx: RunContextWrapper[TenderState]) -> str:
    """Check what has been collected so far and what is still missing."""
    s = ctx.context
    tender = f"LOADED from {s.tender_source} ({len(s.tender_text):,} characters)" if s.tender_text.strip(
    ) else "MISSING"
    company = s.company_profile.strip()[:3000] or "NOTHING SAVED YET"
    return f"TENDER: {tender}\n\nCOMPANY DETAILS SAVED SO FAR:\n{company}"

In [13]:
# The four specialists
def analyst_instructions(ctx: RunContextWrapper[TenderState], agent: Agent) -> str:
    return f"""You are a tender Requirements Analyst. Extract what the tender actually says.

RULES
- Only extract things the tender explicitly states. Never fill gaps from general knowledge.
- Every requirement and deadline needs source_page (use the === PAGE n === markers) and a
  source_quote: a short exact quote (max 25 words) copied word for word from the tender.
- mandatory = true only if the tender uses mandatory language (must, shall, required, will be rejected).
- If something is not stated, write "Not stated". Put vague or missing information in unclear_or_missing_info.
- Capture how bids are scored (evaluation_criteria), including weights and pass marks.
- Copy dates exactly as written. Do not convert them or assume time zones.
- If the document does not look like a tender, return empty lists and say so in the summary.

TENDER TEXT
{ctx.context.tender_text}"""


def requirements_digest(a: TenderAnalysis) -> str:
    groups = [("eligibility", a.eligibility), ("documentation", a.mandatory_documents),
              ("technical", a.technical_requirements), ("commercial", a.commercial_requirements)]
    rows = [f"- [{label}] {'MANDATORY' if r.mandatory else 'optional'}: {r.description}"
            for label, group in groups for r in group]
    return "\n".join(rows) or "(no requirements found)"


def gap_instructions(ctx: RunContextWrapper[TenderState], agent: Agent) -> str:
    analysis: TenderAnalysis = ctx.context.results["analysis"]
    profile = ctx.context.company_profile.strip() or "(NO COMPANY DETAILS SUPPLIED)"
    return f"""You are a tender Gap Reviewer. Compare the tender requirements with the company details.

For EVERY requirement below choose a status:
- evidence_supports: the details contain specific evidence. Put a short exact quote from them in `evidence`.
- partially_supported: some evidence, but not all.
- missing_evidence: the details say nothing either way.
- not_met: the details clearly show it is not met.
- unclear: you cannot tell.

RULES
- Never say a company "is compliant". A claim is not proof; say which document must still be attached.
- Use evidence_supports ONLY if you can quote the company details word for word. Otherwise use missing_evidence.
- Never invent facts about the company.
- Also write clarification_questions: things the company should ask the tender issuer, especially
  vague wording, conflicting dates and missing information.
- If no company details were supplied, mark everything missing_evidence and set overall_readiness to cannot_assess.

REQUIREMENTS
{requirements_digest(analysis)}

TENDER UNCLEAR POINTS
{chr(10).join('- ' + u for u in analysis.unclear_or_missing_info) or '(none)'}

COMPANY DETAILS
{profile}"""


def risk_instructions(ctx: RunContextWrapper[TenderState], agent: Agent) -> str:
    gaps: GapAssessment = ctx.context.results["gaps"]
    gap_lines = "\n".join(
        f"- {i.requirement} ({i.status})" for i in gaps.items if i.status != "evidence_supports") or "(none)"
    return f"""You are a tender Risk Analyst. Find risks, unclear clauses and conditions that need attention.

RULES
- origin = "stated_in_tender" only when the tender says it directly (penalties, payment terms, guarantees).
  Use "ai_inference" for your own concerns. Be honest about which is which.
- Give source_page when you can, otherwise "Not stated".
- Look for: vague wording, conflicting dates, tight deadlines, penalties, late payment, bonds and guarantees,
  liability, unclear scoring, and the compliance gaps listed below.
- Do not give legal advice. Say where a lawyer or procurement specialist should look.

KNOWN COMPLIANCE GAPS
{gap_lines}

TENDER TEXT
{ctx.context.tender_text}"""


def solution_instructions(ctx: RunContextWrapper[TenderState], agent: Agent) -> str:
    r = ctx.context.results
    analysis: TenderAnalysis = r["analysis"]
    gaps: GapAssessment = r["gaps"]
    risks: RiskAssessment = r["risks"]
    return f"""You are a Bid Solution Advisor. Recommend how this company could meet the tender
technically and commercially, and score well.

RULES
- Build on the tender requirements, the scoring criteria, and what the company details say it can do.
- NEVER claim the company has a capability, product, certificate or partner that its details do not mention.
  Where there is a gap, suggest a way to close it (partner, subcontractor, certification, alternative evidence)
  and say what the company must confirm in needs_from_company.
- Technical: approach, solution design, delivery plan, quality, support and warranty.
- Commercial: pricing approach, payment and cash flow, bonds and guarantees, price validity, risk sharing,
  value-adds. Do not invent prices or figures.
- Say which scoring criterion or risk each recommendation helps with (addresses).
- List your assumptions. These are suggestions for the team to check, not guarantees.

REQUIREMENTS
{requirements_digest(analysis)}

SCORING CRITERIA
{chr(10).join('- ' + c for c in analysis.evaluation_criteria) or '(not stated)'}

COMPLIANCE GAPS
{chr(10).join(f'- {i.requirement} ({i.status})' for i in gaps.items if i.status != 'evidence_supports') or '(none)'}

RISKS
{chr(10).join(f'- [{x.severity}] {x.title}' for x in risks.risks) or '(none)'}

COMPANY DETAILS
{ctx.context.company_profile.strip() or '(none supplied)'}"""


settings = ModelSettings(temperature=0.1)

requirements_analyst = Agent[TenderState](
    name="Requirements Analyst", instructions=analyst_instructions,
    model=MODEL, model_settings=settings, output_type=TenderAnalysis)

gap_reviewer = Agent[TenderState](
    name="Gap Reviewer", instructions=gap_instructions,
    model=MODEL, model_settings=settings, output_type=GapAssessment)

risk_analyst = Agent[TenderState](
    name="Risk Analyst", instructions=risk_instructions,
    model=MODEL, model_settings=settings, output_type=RiskAssessment)

solution_advisor = Agent[TenderState](
    name="Solution Advisor", instructions=solution_instructions,
    model=MODEL, model_settings=ModelSettings(temperature=0.3), output_type=SolutionAdvice)

In [14]:
# Agents as tools, the Manager, and the Strategist
def specialist_tool(agent, tool_name, description, key, digest, requires=None, postprocess=None):
    """Wrap an agent as a tool. The full typed result is saved in the shared notebook;
    the Manager only gets a short summary line."""

    @function_tool(name_override=tool_name, description_override=description)
    async def run_specialist(ctx: RunContextWrapper[TenderState], focus: str) -> str:
        """Run a specialist agent.

        Args:
            focus: One short sentence on what to focus on. The documents are already available to the specialist.
        """
        if requires and requires not in ctx.context.results:
            return f"ERROR: the step that produces '{requires}' has not run yet. Run it first."
        result = await Runner.run(agent, f"Task from the manager: {focus}", context=ctx.context, max_turns=3)
        output = result.final_output
        if postprocess:
            output = postprocess(output, ctx.context)
        ctx.context.results[key] = output
        return digest(output)

    return run_specialist


def analysis_digest(a: TenderAnalysis) -> str:
    n = sum(len(g) for g in (a.eligibility, a.mandatory_documents,
            a.technical_requirements, a.commercial_requirements))
    warn = " WARNING: nothing tender-like was found; this may not be a tender." if n == 0 and not a.deadlines else ""
    return f"Requirements analysis complete: '{a.tender_title}', {n} requirements, {len(a.deadlines)} deadlines.{warn}"


def gap_digest(g: GapAssessment) -> str:
    return f"Gap review complete: {len(g.items)} items checked, readiness: {g.overall_readiness}."


def risk_digest(r: RiskAssessment) -> str:
    return f"Risk analysis complete: {len(r.risks)} risks, {sum(1 for x in r.risks if x.severity == 'high')} high severity."


def solution_digest(s: SolutionAdvice) -> str:
    return f"Solution advice complete: {len(s.technical)} technical, {len(s.commercial)} commercial, {len(s.teaming_or_subcontracting)} teaming ideas."


tools = [
    specialist_tool(requirements_analyst, "requirements_analyst",
                    "Reads the tender and extracts requirements, deadlines, eligibility, scoring and submission rules. Run FIRST.",
                    "analysis", analysis_digest),
    specialist_tool(gap_reviewer, "gap_reviewer",
                    "Compares tender requirements with the company details and lists gaps and clarification questions. Run SECOND.",
                    "gaps", gap_digest, requires="analysis",
                    postprocess=lambda out, st: enforce_evidence_rules(out, st.company_profile)),
    specialist_tool(risk_analyst, "risk_analyst",
                    "Finds risks, unclear clauses and conditions needing attention. Run THIRD.",
                    "risks", risk_digest, requires="gaps"),
    specialist_tool(solution_advisor, "solution_advisor",
                    "Recommends technical and commercial solutions and teaming options. Run FOURTH.",
                    "solutions", solution_digest, requires="risks"),
]


def strategist_instructions(ctx: RunContextWrapper[TenderState], agent: Agent) -> str:
    r = ctx.context.results
    return f"""{RECOMMENDED_PROMPT_PREFIX}
You are the Bid Strategist. The analysis is finished and you now take over. Using ONLY the results
below, write the final TenderWise AI report in Markdown, with these sections in this order:

# TenderWise AI Report: <tender title>
(One line: this is AI-generated help, not legal or procurement advice. It does not certify compliance.)
## 1. Bid decision snapshot - a preliminary view (Proceed / Proceed with conditions / Do not bid), 3 to 5 bullets of reasons, and what it depends on
## 2. Tender overview
## 3. Eligibility criteria - a table: requirement | mandatory? | status for this company | page
## 4. Critical submission requirements and deadlines
## 5. Gaps and clarification questions
## 6. Risks - most serious first
## 7. Recommended technical solution
## 8. Recommended commercial approach
## 9. Winning strategy - 3 to 5 win themes, how to position the company, how to strengthen the bid, and where the scoring points are
## 10. Proposal outline
## 11. Submission checklist - as - [ ] items
## 12. Next 7 days - a short action plan

LABELS: mark items as 📄 Stated in tender, 🟠 Missing evidence, or 🤖 AI recommendation or concern.
RULES
- Never say the company "is compliant". Say what evidence exists and what must still be attached.
- Do not add facts that are not in the results. Do not invent prices, dates or capabilities.
- Be concise. Use tables and bullets.
- If the conversation already contains your report, answer follow-up questions briefly using these results.
  Do not rewrite the whole report.

TENDER ANALYSIS
{r["analysis"].model_dump_json()}

GAP ASSESSMENT
{r["gaps"].model_dump_json()}

RISK ASSESSMENT
{r["risks"].model_dump_json()}

SOLUTION ADVICE
{r["solutions"].model_dump_json()}"""


strategist = Agent[TenderState](
    name="Bid Strategist",
    handoff_description="Takes over when the analysis is done and writes the final report and bid strategy.",
    instructions=strategist_instructions,
    model=MODEL, model_settings=ModelSettings(temperature=0.3),
)

manager = Agent[TenderState](
    name="Tender Intelligence Manager",
    handoff_description="Runs the full tender analysis once the tender and company details have been collected.",
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
You are the Tender Intelligence Manager. You coordinate; you do not analyse the tender yourself.

Do these steps IN ORDER, one tool call at a time, waiting for each result:
1. requirements_analyst
2. gap_reviewer
3. risk_analyst
4. solution_advisor
5. When all four are finished, hand off to the Bid Strategist.

If requirements_analyst warns this may not be a tender, stop and explain in one or two sentences instead of
continuing. Never skip a step and never write the final report yourself.""",
    tools=tools,
    handoffs=[strategist],
    model=MODEL,
    model_settings=ModelSettings(temperature=0, parallel_tool_calls=False),
)

In [15]:
# The Intake Agent (Asks you to attach files)
intake_agent = Agent[TenderState](
    name="Intake Agent",
    instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
You are the TenderWise AI intake assistant. Before any analysis, collect two things from the user, inside a chat
window that lets them attach files. Be warm, short and plain. Ask for one step at a time.
The chat window has already greeted the user and asked them to attach the tender document.

HOW ATTACHMENTS WORK
When the user attaches a file you will see a note like: [The user attached the file 'x.pdf' (file_id: upload_1)].
Use that file_id with the tools. Never ask the user to type a file path. Never guess a file_id.
If you cannot tell whether an attached file is the tender or the company profile, ask.

STEP 1 - THE TENDER
If the user has not attached the tender yet, ask them to attach it (PDF, Word or TXT) with the 📎 button.
When they do, call load_tender_file. If they type tender details instead, call save_tender_text.
If loading fails, explain what went wrong and ask again.

STEP 2 - THE COMPANY AND ITS EXPERTISE FOR THIS OPPORTUNITY
Once the tender is loaded, say so in one sentence and ask about the company, at most 3 questions at a time:
- Legal registration: company name, country, year started, tax clearance
- Relevant experience: similar projects (client, year, value)
- Financial capacity: annual turnover, ability to provide bonds or guarantees
- Certifications and licences
- Key staff and qualifications
- Partners, manufacturer authorisations, subcontractors
- What the company would offer for THIS tender (products, services, approach) and what makes it stand out
Tell the user they can also attach a company profile or capability statement instead of typing; then call
load_company_file. After each typed answer call save_company_details with the user's own words.
Never add, guess or improve facts. If the user does not have something, save "<topic>: not provided by user".

STEP 3 - START
When the tender is loaded and you have the main company details (or the user says that is all),
call check_intake_status. If the tender is loaded, tell the user in one sentence that analysis is starting and
may take a minute or two, then hand off to the Tender Intelligence Manager.
If the tender is missing, ask for it. Never start the analysis without a tender.""",
    tools=[load_tender_file, save_tender_text, save_company_details,
           load_company_file, check_intake_status],
    handoffs=[manager],
    model=MODEL,
    model_settings=ModelSettings(temperature=0.2),
)

In [16]:
# The Gradio chat
GREETING = (
    "👋 Hi, I'm **TenderWise AI**. I help you understand a tender and plan a stronger bid.\n\n"
    "**Step 1:** attach the tender document (PDF, Word or TXT) with the 📎 button below, then press send."
)

STEP_LABELS = {
    "transfer_to_tender_intelligence_manager": "Handing your details to the Tender Intelligence Manager",
    "requirements_analyst": "Requirements Analyst is reading the tender",
    "gap_reviewer": "Gap Reviewer is checking your company against the requirements",
    "risk_analyst": "Risk Analyst is looking for risks and unclear clauses",
    "solution_advisor": "Solution Advisor is working on technical and commercial recommendations",
    "transfer_to_bid_strategist": "Bid Strategist is writing your report",
}

In [17]:
class Session:
    """Remembers the conversation. One user at a time (fine for a local tool)."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.state = TenderState()
        self.agent = intake_agent
        self.messages = []
        self.reported = False


SESSION = Session()


def file_path(f) -> str:
    """Gradio can hand us a plain path or a small dictionary, depending on the version."""
    if isinstance(f, str):
        return f
    if isinstance(f, dict):
        return f.get("path") or f.get("name") or ""
    return getattr(f, "path", None) or str(f)


def verification_notes(st: TenderState) -> str:
    lines = ["\n\n---\n## Verification notes (checked by code, not by the AI)"]
    cut = " (the tender was long and was cut at the size limit)" if st.truncated else ""
    lines.append(
        f"- Tender source: {st.tender_source}; pages read: {st.pages}{cut}")
    bad = find_unverified_quotes(st.results["analysis"], st.tender_text)
    if bad:
        lines.append(
            "- These quotes could not be found word for word in the tender text. Check them by hand:")
        lines += [f"  - {b}" for b in bad]
    else:
        lines.append(
            "- Every quoted requirement and deadline was found in the tender text.")
    return "\n".join(lines)

In [18]:
async def chat(message, history):
    if len(history) <= 1:            # a fresh chat (or the chat was cleared)
        SESSION.reset()
    s = SESSION

    if isinstance(message, str):
        message = {"text": message, "files": []}
    text = (message.get("text") or "").strip()

    # Register attached files under short ids, and tell the agent about them
    notes = []
    for f in message.get("files") or []:
        p = Path(file_path(f))
        file_id = f"upload_{len(s.state.uploads) + 1}"
        s.state.uploads[file_id] = p
        notes.append(
            f"[The user attached the file '{p.name}' (file_id: {file_id})]")
    content = "\n".join([text] + notes).strip() or "Hello"

    try:
        run = Runner.run_streamed(
            s.agent,
            s.messages + [{"role": "user", "content": content}],
            context=s.state,
            max_turns=30,
        )
        steps = []
        async for event in run.stream_events():     # show progress while the team works
            if event.type == "run_item_stream_event":
                name = getattr(event.item.raw_item, "name", None)
                if name in STEP_LABELS and STEP_LABELS[name] not in steps:
                    steps.append(STEP_LABELS[name])
                    yield "⏳ **Working on it...** (this can take a minute or two)\n\n" + "\n".join(f"- {x}" for x in steps)
    except Exception as e:
        yield f"Something went wrong: {type(e).__name__}: {e}"
        return

    s.messages = run.to_input_list()
    s.agent = run.last_agent
    reply = run.final_output

    if s.agent.name == "Bid Strategist" and not s.reported:     # the report is ready
        s.reported = True
        reply = reply + verification_notes(s.state)
        Path("output").mkdir(exist_ok=True)
        Path("output/tenderwise_report.md").write_text(reply, encoding="utf-8")
        reply += "\n\n_Saved to `output/tenderwise_report.md`. Ask me follow-up questions, for example how to price this bid._"
    # the manager stopped early (not a tender?)
    elif s.agent is manager:
        s.agent = intake_agent

    yield reply

In [19]:
demo = gr.ChatInterface(
    chat,
    multimodal=True,
    chatbot=gr.Chatbot(
        value=[{"role": "assistant", "content": GREETING}], height=600),
    textbox=gr.MultimodalTextbox(
        file_types=[".pdf", ".txt", ".docx"],
        file_count="multiple",
        placeholder="Type here, or attach a file with the 📎 button...",
    ),
    title="TenderWise AI",
    description="Upload a tender, tell me about your company, and get a bid strategy. AI-generated help, not legal or procurement advice.",
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
